In [2]:
# 1. Cài đặt thư viện PySpark
!pip install pyspark -q

# 2. Khởi tạo Spark Session tối ưu cho môi trường Kaggle
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Instacart_BigData_Analysis") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .config("spark.sql.repl.eagerEval.maxNumRows", 10) \
    .getOrCreate()

print("Cài đặt Spark Session thành công!")

Cài đặt Spark Session thành công!


In [3]:
import pyspark.sql.functions as F

# Đường dẫn bộ dữ liệu Instacart trên hệ thống Kaggle
base_path = "/kaggle/input/datasets/yasserh/instacart-online-grocery-basket-analysis-dataset/"

# Đọc các bảng dữ liệu
df_orders = spark.read.csv(base_path + "orders.csv", header=True, inferSchema=True)
df_order_products = spark.read.csv(base_path + "order_products__prior.csv", header=True, inferSchema=True)
df_products = spark.read.csv(base_path + "products.csv", header=True, inferSchema=True)
df_aisles = spark.read.csv(base_path + "aisles.csv", header=True, inferSchema=True)
df_departments = spark.read.csv(base_path + "departments.csv", header=True, inferSchema=True)


# In kích thước thực tế 
print(f"[SHAPE] Bảng Orders gốc: ({df_orders.count()} dòng, {len(df_orders.columns)} cột)")
print(f"[SHAPE] Bảng Order Products gốc: ({df_order_products.count()} dòng, {len(df_order_products.columns)} cột)")
print(f"[SHAPE] Bảng Products: ({df_products.count()} dòng)")
print(f"[SHAPE] Bảng Aisles: ({df_aisles.count()} dòng)")
print(f"[SHAPE] Bảng Departments: ({df_departments.count()} dòng)")


[SHAPE] Bảng Orders gốc: (3421083 dòng, 7 cột)


[SHAPE] Bảng Order Products gốc: (32434489 dòng, 4 cột)
[SHAPE] Bảng Products: (49688 dòng)
[SHAPE] Bảng Aisles: (134 dòng)
[SHAPE] Bảng Departments: (21 dòng)


In [5]:
df_orders.createOrReplaceTempView("orders")
df_order_products.createOrReplaceTempView("order_products")
df_products.createOrReplaceTempView("products")
df_aisles.createOrReplaceTempView("aisles")
df_departments.createOrReplaceTempView("departments")

print("Đăng ký Temp Views hoàn tất cho các bảng Instacart!")

Đăng ký Temp Views hoàn tất cho các bảng Instacart!


In [6]:
print('Dữ liệu bảng Orders')
spark.sql("""
    SELECT *
    FROM orders
    ORDER BY user_id DESC
    LIMIT 5
""").show(truncate=False)

print('\nDữ liệu bảng Order Products')
spark.sql("""
    SELECT *
    FROM order_products
    LIMIT 5
""").show(truncate=False)

print('\nDữ liệu bảng Products')
spark.sql("""
    SELECT *
    FROM products
    LIMIT 5
""").show(truncate=False)

print('\nDữ liệu bảng Aisles')
spark.sql("""
    SELECT *
    FROM aisles
    LIMIT 5
""").show(truncate=False)

print('\nDữ liệu bảng Departments')
spark.sql("""
    SELECT *
    FROM departments
    LIMIT 5
""").show(truncate=False)

Dữ liệu bảng Orders


+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
|3154581 |206209 |prior   |1           |3        |11               |NULL                  |
|2307371 |206209 |prior   |5           |4        |15               |3.0                   |
|1889163 |206209 |prior   |2           |3        |17               |7.0                   |
|1542354 |206209 |prior   |3           |5        |11               |30.0                  |
|688306  |206209 |prior   |4           |1        |10               |30.0                  |
+--------+-------+--------+------------+---------+-----------------+----------------------+


Dữ liệu bảng Order Products
+--------+----------+-----------------+---------+
|order_id|product_id|add_to_cart_order|reordered|
+--------+----------+-----------------+---